In [ ]:
import jax
import jax.numpy as jnp
from jax import lax, jit
from jax.experimental import mesh_utils
from jax.sharding import PositionalSharding
from functools import partial
import time

# --- Configuration ---
MAX_RECURSION_DEPTH    = 1_000_000
OPTIMAL_DEPTH_STEP     = 250_000
DIMENSIONAL_CONSTRAINT = 0.8
BATCH_SIZE            = 50_000_000

@jit
def dynamic_pi(depth, scale_factor):
    depth = jnp.minimum(depth, MAX_RECURSION_DEPTH)
    return jnp.pi * jnp.log1p(depth + 1) * scale_factor * DIMENSIONAL_CONSTRAINT

@jit
def dynamic_phi(depth, scale_factor):
    depth = jnp.minimum(depth, MAX_RECURSION_DEPTH)
    return (1 + jnp.sqrt(5)) / 2 * jnp.exp(-depth / (scale_factor + 1)) * DIMENSIONAL_CONSTRAINT

@jit
def stabilize_depth(depth):
    """Normalizes depth scaling to prevent instability."""
    return depth / (1 + jnp.log1p(depth + 1))

# -------------------------------------------------------------------------
# Single chunk of 250k steps, identical to your original dppu_with_dynamic_pi_phi,
# but specialized here so we can embed it in one bigger JIT.
# -------------------------------------------------------------------------
def single_chunk_fori_loop(x, scale_factor=1.0):
    """
    Runs 250k steps of your sin/exp recursion in one fori_loop.
    Returns the final 'x' after 250k iterations.
    """

    def body_fn(i, val):
        pi_dyn  = dynamic_pi(i, scale_factor)
        phi_dyn = dynamic_phi(i, scale_factor)
        scale   = jnp.log1p(i + 1) * scale_factor * DIMENSIONAL_CONSTRAINT
        new_val = jnp.sin(val * scale * pi_dyn) * jnp.exp(-val / (phi_dyn + 1))
        return new_val

    # We ensure the iteration count is an int32 for fori_loop
    steps = jnp.int32(OPTIMAL_DEPTH_STEP)
    return lax.fori_loop(0, steps, body_fn, x)

# -------------------------------------------------------------------------
# The chunked function with debug prints AFTER each 250k-step chunk
# -------------------------------------------------------------------------
@partial(jit, static_argnames=["total_depth", "scale_factor"])
def chunked_dppu_debug(x, total_depth, scale_factor=1.0):
    """
    This single JIT-compiled function:
      - Splits total_depth into (total_depth // 250k) chunks.
      - For each chunk, calls single_chunk_fori_loop (250k steps),
        then does a debug print to show intermediate progress.
    """
    # How many 250k chunks do we need?
    iterations = total_depth // OPTIMAL_DEPTH_STEP

    def chunk_body(chunk_idx, val):
        # val is the current 'x' array
        new_val = single_chunk_fori_loop(val, scale_factor=scale_factor)

        # Optional debug metric: let's compute mean of new_val or any statistic
        mean_val = jnp.mean(new_val)
        # jax.debug.print: prints host-side, but inside JIT after chunk
        jax.debug.print(
            "After chunk {chunk_idx}, mean(new_val)={mean_val:.6f}",
            chunk_idx=chunk_idx,
            mean_val=mean_val
        )
        return new_val

    final_x = lax.fori_loop(0, iterations, chunk_body, x)
    return final_x

# -------------------------------------------------------------------------
# Convenience function to call the chunked recursion
# -------------------------------------------------------------------------
def process_with_larger_depths_debug(x, total_depth, scale_factor=1.0):
    """
    High-level Python function that calls the single JIT-compiled chunked recursion.
    """
    return chunked_dppu_debug(x, total_depth=total_depth, scale_factor=scale_factor)

# -------------------------------------------------------------------------
# Main script to test at various depths
# -------------------------------------------------------------------------
if __name__ == "__main__":

    # TPU/Device Sharding Setup (if you have multiple devices)
    devices = jax.devices()
    sharding = PositionalSharding(devices)

    # Create batch input and put on device
    batch_input = jnp.linspace(0, 10, BATCH_SIZE)
    batch_input = jax.device_put(batch_input, sharding)

    # Warm-up compile at 250k
    _ = chunked_dppu_debug(batch_input, total_depth=250_000, scale_factor=0.5)

    # Try different total depths
    for depth in [250_000, 500_000, 1_000_000]:
        start_time = time.time()
        output = process_with_larger_depths_debug(batch_input, depth, scale_factor=0.5)
        # Force blocking on device
        output_host = jax.device_get(output)
        end_time = time.time()

        # Final stats
        mean_final = jnp.mean(output_host)
        print(f"\nDepth={depth}, final mean={mean_final:.6f}")
        print(f"Output shape: {output_host.shape}")
        print(f"Time: {end_time - start_time:.4f} sec")


After chunk 0, mean(new_val)=nan
After chunk 0, mean(new_val)=nan

Depth=250000, final mean=nan
Output shape: (50000000,)
Time: 640.9527 sec
After chunk 0, mean(new_val)=nan
After chunk 1, mean(new_val)=nan

Depth=500000, final mean=nan
Output shape: (50000000,)
Time: 641.5221 sec
After chunk 0, mean(new_val)=nan
After chunk 1, mean(new_val)=nan
After chunk 2, mean(new_val)=nan
After chunk 3, mean(new_val)=nan

Depth=1000000, final mean=nan
Output shape: (50000000,)
Time: 1282.2792 sec
